In [4]:
# Installation rapide
!pip install polars s3fs boto3 pyarrow pandas openpyxl --quiet


[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: pip install --upgrade pip


# Chargement dans la base concerné

In [ ]:
import boto3
import io
import pandas as pd
from io import BytesIO

# Connexion à MinIO
s3_client = boto3.client(
    "s3",
    endpoint_url="http://minio.mon-namespace.svc.cluster.local:80",
    aws_access_key_id="Aminata",
    aws_secret_access_key="AminataCae312",
    verify=False
)

# Paramètres
bucket_name = "admindataanstat"
object_key = "Solde/DONNEES 2015/012015.xlsx"

# Télécharger le fichier Excel depuis S3
response = s3_client.get_object(Bucket=bucket_name, Key=object_key)
bytes_io = io.BytesIO(response['Body'].read())

# Charger le fichier Excel
excel_file = pd.ExcelFile(bytes_io)

# Afficher les feuilles disponibles
print("Feuilles disponibles dans le fichier Excel :")
print(excel_file.sheet_names)

In [ ]:
s3_client = boto3.client(
 "s3",
 endpoint_url="http://minio.mon-namespace.svc.cluster.local:80", # service interne K8s
 aws_access_key_id="Aminata",
 aws_secret_access_key="AminataCae312",
 verify=False # à désactiver si SSL auto-signé ; sinon mettre True avec certificat valide
)

In [ ]:
bucket_name = "admindataanstat"
object_key = "Solde/DONNEES 2015/012015.xlsx"

## Suppression de l'entête du fichier

In [ ]:
import pandas as pd
from io import BytesIO

obj = s3_client.get_object(Bucket=bucket_name, Key=object_key)
data_bytes = obj['Body'].read()

# Charge d'abord tout dans la mémoire pour réutiliser le même fichier
bio = BytesIO(data_bytes)

# 1. Lire la 2e feuille "brut" (sans entête, pour trouver la vraie entête)
data_tmp = pd.read_excel(bio, sheet_name=1, header=None, engine='openpyxl')

#suprresion de la dernière ligne 
data_tmp = data_tmp.iloc[:-1].reset_index(drop=True)

# Trouver la ligne d'entête contenant "matricule"
header_row = None
for i, row in data_tmp.iterrows():
    row_str = row.astype(str).str.lower().str.strip().tolist()
    if any("matricule" in cell for cell in row_str):
        header_row = i
        break

if header_row is None:
    raise ValueError("Aucune entête avec 'MATRICULE' détectée !")

# 2. Revenir au début du fichier pour relire !
bio.seek(0)
data = pd.read_excel(bio, sheet_name=1, header=header_row, engine='openpyxl')

In [ ]:
data.head()

# Fusion

In [ ]:
# 1. Connexion au serveur
bucket_name = "admindataanstat"
prefixes = ["Solde/DONNEES 2015/", "Solde/DONNEES 2016/", "Solde/DONNEES 2017/", "Solde/DONNEES 2018/", "Solde/DONNEES 2019/", "Solde/DONNEES 2020/"]

s"assurer que dans chaque dossier il ya vait 12 fichiers

In [ ]:
# 2. Récupérer tous les fichiers Excel au format MMYYYY.xlsx ==========
import re

fichiers_par_mois = {}

for prefix in prefixes:
    response = s3_client.list_objects_v2(Bucket=bucket_name, Prefix=prefix)

    if "Contents" in response:
        for obj in response["Contents"]:
            key = obj["Key"]
            match = re.search(r"(\d{6})\.xlsx$", key)  # Ex : 012015
            if match:
                mois = match.group(1)
                fichiers_par_mois.setdefault(mois, []).append(key)

print(f"🔍 Fichiers détectés pour {len(fichiers_par_mois)} périodes.")


In [ ]:
# 3. Charger les fichiers, ajouter la période, et concaténer
resultat_final = []

for mois, fichiers in fichiers_par_mois.items():
    for fichier in fichiers:
        try:
            # Lire le fichier Excel brut sans entête (pour ignorer les lignes vides)
            obj = s3_client.get_object(Bucket=bucket_name, Key=fichier)
            data_bytes = obj['Body'].read()
            df_raw = pd.read_excel(BytesIO(data_bytes), header=None, sheet_name=1)
            df_raw.dropna(how='all', inplace=True)

            # Identifier la bonne ligne d'en-tête
            colonnes_requises = {"MATRICULE||CODE_ORGANISME", "MATRICULE", "NOM", "DATE NAISSANCE","SEXE", "SITUATION MATRIMONIALE", "NOMBRE ENFANT", "LIEU AFFECTATION","SERVICE", "ORGANISME", "PRISE DE SERVICE", "SITUATION", "DATE DEBUT SITUATION", "EMPLOI", "FONCTION", "GRADE", "CLASSE/ECHELON", "MONTANT BRUT", "MONTANT NET", "RETENUE  PENSION", "IMPOT ", "CHARGE PATRONALE", 'MOIS/ANNEE', "GRADE 2", "AGE RETRAITE", "DATE RETRAITE"}
            for i, row in df_raw.iterrows():
                if colonnes_requises.issubset(set(row.astype(str))):
                    header_row_index = i
                    break
            else:
                print(f"⚠️ Aucune ligne d'en-tête détectée dans : {fichier}")
                continue  # ✅ ce `continue` est bien dans la boucle "for fichier in fichiers"

            # Relire avec les bons noms de colonnes
            df = pd.read_excel(BytesIO(data_bytes), skiprows=header_row_index, sheet_name=1)

            # Vérifier présence des colonnes
            if colonnes_requises.issubset(df.columns):
                df["PERIODE"] = mois
                df["DATE_COLLECTE"] = pd.to_datetime(mois, format="%m%Y")
                resultat_final.append(df)
                print(f"✅ {fichier} ajouté.")
            else:
                print(f"⚠️ Colonnes manquantes dans : {fichier} → {colonnes_requises - set(df.columns)}")

        except Exception as e:
            print(f"❌ Erreur avec {fichier} : {e}")

⚠️ Aucune ligne d'en-tête détectée dans : Solde/DONNEES 2015/042015.xlsx
⚠️ Aucune ligne d'en-tête détectée dans : Solde/DONNEES 2015/102015.xlsx
⚠️ Aucune ligne d'en-tête détectée dans : Solde/DONNEES 2015/112015.xlsx


In [15]:
# ========== 4. Création du DataFrame panel ==========
if resultat_final:
    panel_solde_df_2 = pd.concat(resultat_final, ignore_index=True)

    # Supprimer les lignes entièrement vides (optionnel, sécurité)
    panel_solde_df_2.dropna(how='all', inplace=True)

    print("✅ Fusion complète en format panel.")
    print(f"📊 Nombre total de lignes : {len(panel_solde_df_2)}")
    print(panel_solde_df_2.head())

    # (Facultatif) Export vers un fichier local
    # panel_solde_df.to_excel("panel_solde_df.xlsx", index=False)
    # panel_solde_df.to_csv("panel_solde_df.csv", index=False)

else:
    print("❌ Aucun fichier n'a été chargé correctement.")

✅ Fusion complète en format panel.
📊 Nombre total de lignes : 15627963
  MATRICULE||CODE_ORGANISME MATRICULE                        NOM  \
0                 011242X15   011242X                DOSSO MEGBO   
1                 012856Q15   012856Q   KHISSY BEYNIOUAH FULBERT   
2                 013454N15   013454N  VAHA TOMAS GNOMBLEI HENRI   
3                 027861L25   027861L         CAPET YATO MATHIEU   
4                 039659M38   039659M         COULIBALY YOUSSOUF   

        DATE NAISSANCE      SEXE SITUATION MATRIMONIALE  NOMBRE ENFANT  \
0  1939-01-01 00:00:00  Masculin                  Marié            0.0   
1  1939-01-01 00:00:00  Masculin            Célibataire            0.0   
2  1924-01-01 00:00:00  Masculin                  Marié            0.0   
3  1943-03-01 00:00:00  Masculin                  Marié            0.0   
4  1945-12-01 00:00:00  Masculin                  Marié            2.0   

  LIEU AFFECTATION                         SERVICE  \
0          Seguela   

# Enregistrement de la base sous format CSV

In [18]:
import io

# Convertir en CSV
csv_buffer = io.StringIO()
panel_solde_df.to_csv(csv_buffer, index=False, encoding="utf-8-sig")

# Sauvegarder dans le bucket S3
s3_client.put_object(
    Bucket="admindataanstat",
    Key="Solde/panel_solde_df_2.csv",
    Body=csv_buffer.getvalue()
)

print("✅ Ta base a été sauvegardée dans le bucket S3 : Solde/panel_solde_df_2.csv")

✅ Ta base a été sauvegardée dans le bucket S3 : Solde/panel_solde_df.csv
